# SQL Schema Design — Complete Reference

| Pattern | Concept |
|---------|----------|
| Normal forms | 1NF / 2NF / 3NF — eliminate redundancy |
| BCNF + denormalization | When to break normal forms deliberately |
| Natural vs surrogate keys | Trade-offs for primary key selection |
| Wide tables | Column-per-attribute vs EAV vs JSON |
| Audit / temporal tables | Soft delete, history tracking, SCD Type 2 |

**Mental model**: Normalization removes redundancy by ensuring each fact is stored in exactly one place. Denormalization adds redundancy deliberately for read performance. Good schema design knows when each is appropriate.

```
Normal Forms (progressive):
  1NF: atomic values, no repeating groups, primary key
  2NF: 1NF + no partial dependency on composite key
  3NF: 2NF + no transitive dependency (non-key depends only on key)
  BCNF: 3NF + every determinant is a candidate key
```

## Visual Model

```
NORMALIZATION PROGRESSION
───────────────────────────
Denormalized (0NF):
  order_id | customer | customer_email | product1 | qty1 | product2 | qty2
  1001     | Alice    | a@e.com        | Widget   | 2    | Gadget   | 1
  Problem: repeating groups (product1/qty1, product2/qty2), update anomaly

1NF: atomic values, no repeating groups
  order_id | customer | email   | product | qty
  1001     | Alice    | a@e.com | Widget  | 2
  1001     | Alice    | a@e.com | Gadget  | 1
  Problem: customer/email repeated for every order line

2NF: eliminate partial dependency (product_name depends only on product_id)
  orders(order_id, customer_id)   order_items(order_id, product_id, qty)
  products(product_id, name)      customers(customer_id, name, email)

3NF: eliminate transitive dependency
  If customer_id → zip_code → city, then city depends transitively on customer_id
  Fix: zip_codes(zip, city, state) table; customers(customer_id, zip)

SCD TYPE 2 (Slowly Changing Dimension)
────────────────────────────────────────
  Keep history of changes:
  id | customer_id | tier   | valid_from | valid_to   | is_current
  1  | 101         | Silver | 2022-01-01 | 2023-06-15 | 0
  2  | 101         | Gold   | 2023-06-15 | NULL       | 1  ← current
```

## Setup — Libraries and Config

In [ ]:
import sqlite3

print(f"sqlite3 version: {sqlite3.sqlite_version}")

def make_db():
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

def run(conn, sql, title=""):
    try:
        cur = conn.execute(sql)
    except Exception as e:
        print(f"  ERROR: {e}")
        return
    if cur.description is None:
        print(f"{title}: (no result set)")
        return
    rows = cur.fetchall()
    if not rows:
        print(f"{title}: (no rows)")
        return
    cols = [d[0] for d in cur.description]
    col_w = [max(len(c), max(len(str(r[c])) for r in rows)) for c in cols]
    sep = "  ".join("-" * w for w in col_w)
    header = "  ".join(c.ljust(w) for c, w in zip(cols, col_w))
    if title:
        print(f"\n=== {title} ===")
    print(header)
    print(sep)
    for r in rows:
        print("  ".join(str(r[c]).ljust(w) for c, w in zip(cols, col_w)))

conn = make_db()
print("Database ready.")

## Decision Map — Schema Design Choices

```
Starting from raw data — what to normalize?
│
├─ Repeating groups in one row?          → 1NF: separate into rows
├─ Non-key col depends on part of key?  → 2NF: extract to own table
├─ Non-key col depends on non-key col?  → 3NF: extract dependency chain
└─ Normalized but queries are slow?     → Denormalize selectively

Primary key choice:
  Natural key (email, SSN):  meaningful, enforces uniqueness, but can change
  Surrogate key (auto int):  stable, small, simple joins, no business meaning
  UUID:                      globally unique, good for distributed, larger storage
  Composite natural:         use when combination defines uniqueness (order_id+line)

Wide table vs normalized:
  Wide table:     few tables, fast reads, OK for analytics (star schema facts)
  Normalized:     many tables, transactional integrity, no redundancy
  EAV:            flexible schema, terrible query performance — avoid for core data
  JSON column:    semi-structured extras, not queryable without extraction

Audit / change history:
  Soft delete:    is_deleted flag + deleted_at timestamp
  Append-only:    new row per change (event sourcing style)
  SCD Type 2:     valid_from / valid_to + is_current flag (dim table pattern)
  Audit table:    separate audit_log table with before/after snapshots
```

## Pattern 1 — 1NF / 2NF / 3NF

In [ ]:
# Demonstrate each normal form with a concrete before/after

# 0NF / Denormalized (violates 1NF: multi-value columns)
conn.executescript("""
    -- 0NF: multi-value product_ids column (not atomic)
    CREATE TABLE orders_bad (
        order_id INTEGER,
        customer_name TEXT,
        customer_email TEXT,
        product_ids TEXT,   -- '10,20,30' — violates 1NF
        zip_code TEXT,
        city TEXT           -- transitive: zip → city (violates 3NF)
    );
    INSERT INTO orders_bad VALUES
        (1001,'Alice','a@e.com','10,20',  '10001','New York'),
        (1002,'Alice','a@e.com','30',     '10001','New York'),
        (1003,'Bob',  'b@e.com','10',     '90001','Los Angeles');
""")
run(conn, "SELECT * FROM orders_bad", "0NF (bad): multi-value column + repeated customer")

# 3NF proper schema
conn.executescript("""
    -- 3NF: separate tables, no partial/transitive dependencies
    CREATE TABLE zip_codes   (zip TEXT PRIMARY KEY, city TEXT, state TEXT);
    CREATE TABLE customers   (customer_id INTEGER PRIMARY KEY, name TEXT, email TEXT, zip TEXT
                              REFERENCES zip_codes(zip));
    CREATE TABLE products    (product_id INTEGER PRIMARY KEY, name TEXT, price REAL);
    CREATE TABLE orders_3nf  (order_id INTEGER PRIMARY KEY, customer_id INTEGER
                              REFERENCES customers(customer_id), order_date TEXT);
    CREATE TABLE order_items (order_id INTEGER REFERENCES orders_3nf(order_id),
                              product_id INTEGER REFERENCES products(product_id),
                              qty INTEGER, PRIMARY KEY (order_id, product_id));

    INSERT INTO zip_codes VALUES ('10001','New York','NY'),('90001','Los Angeles','CA');
    INSERT INTO customers VALUES (1,'Alice','a@e.com','10001'),(2,'Bob','b@e.com','90001');
    INSERT INTO products VALUES (10,'Widget',250),(20,'Gadget',180),(30,'Drill',320);
    INSERT INTO orders_3nf VALUES (1001,1,'2024-01-05'),(1002,1,'2024-01-20'),(1003,2,'2024-01-08');
    INSERT INTO order_items VALUES (1001,10,2),(1001,20,1),(1002,30,1),(1003,10,1);
""")

run(conn, """
    SELECT o.order_id, c.name, z.city, p.name AS product, oi.qty, p.price * oi.qty AS line_total
    FROM orders_3nf o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN zip_codes z ON c.zip = z.zip
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    ORDER BY o.order_id, p.name
""", "3NF schema: reconstructed with joins")

print("""
3NF benefits:
  Change Alice's email: UPDATE customers SET email='new@e.com' WHERE customer_id=1
  Change city for zip:  UPDATE zip_codes SET city='NYC' WHERE zip='10001'
  No redundant updates needed — each fact in one place
""")

## Pattern 2 — BCNF + Deliberate Denormalization

In [ ]:
# BCNF: every determinant must be a candidate key
# Denormalization: add redundancy for read speed, at cost of update complexity

# Example: BCNF violation
# teaching(student, subject, teacher)  FK: teacher → subject (teacher teaches one subject)
# But student+subject is the key — teacher depends on subject, not on key
# Fix: split into (subject, teacher) + (student, subject)

conn.executescript("""
    -- BCNF violation: teacher functionally determines subject
    CREATE TABLE teaching_bad (
        student TEXT, subject TEXT, teacher TEXT,
        PRIMARY KEY (student, subject)
    );
    INSERT INTO teaching_bad VALUES
        ('Alice','Math','Prof Smith'),
        ('Bob','Math','Prof Smith'),
        ('Alice','Physics','Prof Jones');

    -- BCNF fix: separate teacher-subject assignment
    CREATE TABLE subject_teachers (teacher TEXT, subject TEXT, PRIMARY KEY (teacher));
    CREATE TABLE student_enrollments (student TEXT, subject TEXT, PRIMARY KEY (student, subject));
    INSERT INTO subject_teachers VALUES ('Prof Smith','Math'),('Prof Jones','Physics');
    INSERT INTO student_enrollments VALUES ('Alice','Math'),('Bob','Math'),('Alice','Physics');
""")

run(conn, "SELECT * FROM teaching_bad", "BCNF violation: teacher repeated for each student")

# Deliberate denormalization
conn.executescript("""
    -- Denormalized: add region to orders for fast reporting without join
    CREATE TABLE orders_denorm (
        order_id    INTEGER PRIMARY KEY,
        customer_id INTEGER,
        customer_region TEXT,  -- denormalized from customers table
        amount      REAL,
        order_date  TEXT
    );
    INSERT INTO orders_denorm VALUES
        (1,1,'East',250,'2024-01-05'),
        (2,1,'East',180,'2024-01-20'),
        (3,2,'West',500,'2024-01-08');
""")

run(conn, """
    SELECT customer_region, SUM(amount) AS total
    FROM orders_denorm
    GROUP BY customer_region
""", "Denormalized: region aggregation without join")

print("""
Denormalization trade-off:
  READ:   faster (no join for region-based reporting)
  WRITE:  must update all order rows if customer changes region
  Best for: analytical tables (star schema), reporting aggregates, rarely-changing attrs
  Avoid:  operational tables where consistency is critical
""")

## Pattern 3 — Natural vs Surrogate Keys

In [ ]:
# Natural key: real-world identifier (email, SSN, product_code)
# Surrogate key: system-generated integer or UUID (no business meaning)

conn2 = make_db()
conn2.executescript("""
    -- Natural key: email is the PK
    CREATE TABLE customers_natural (
        email TEXT PRIMARY KEY,
        name TEXT,
        tier TEXT
    );
    CREATE TABLE orders_natural (
        order_id INTEGER PRIMARY KEY,
        customer_email TEXT REFERENCES customers_natural(email),
        amount REAL
    );
    INSERT INTO customers_natural VALUES ('a@e.com','Alice','Gold'),('b@e.com','Bob','Silver');
    INSERT INTO orders_natural VALUES (1,'a@e.com',250),(2,'a@e.com',180),(3,'b@e.com',500);

    -- Surrogate key: integer PK, email stored but not the FK
    CREATE TABLE customers_surrogate (
        customer_id INTEGER PRIMARY KEY,
        email TEXT UNIQUE,   -- still unique, just not the FK
        name TEXT,
        tier TEXT
    );
    CREATE TABLE orders_surrogate (
        order_id INTEGER PRIMARY KEY,
        customer_id INTEGER REFERENCES customers_surrogate(customer_id),
        amount REAL
    );
    INSERT INTO customers_surrogate VALUES (1,'a@e.com','Alice','Gold'),(2,'b@e.com','Bob','Silver');
    INSERT INTO orders_surrogate VALUES (1,1,250),(2,1,180),(3,2,500);
""")

print("=== Natural key: email change requires cascade update ===")
print("UPDATE customers_natural SET email='new@e.com' WHERE email='a@e.com'")
print("→ Also must update orders_natural.customer_email (cascade or ON UPDATE CASCADE)")

print("\n=== Surrogate key: customer_id is stable across email changes ===")
print("UPDATE customers_surrogate SET email='new@e.com' WHERE customer_id=1")
print("→ orders_surrogate.customer_id unchanged — no cascade needed")

# Composite natural key example
conn2.executescript("""
    CREATE TABLE order_items (
        order_id   INTEGER,
        product_id INTEGER,
        qty        INTEGER,
        PRIMARY KEY (order_id, product_id)  -- composite natural key
    );
    INSERT INTO order_items VALUES (1,10,2),(1,20,1),(2,30,3);
""")
run(conn2, "SELECT * FROM order_items", "Composite natural key: (order_id, product_id)")

print("""
Key selection guide:
  Surrogate int:  best for OLTP; stable, small FK, index-efficient
  Natural key:    use when business enforces uniqueness (SKU, ISBN, SSN)
  UUID:           distributed systems; globally unique; larger (16 bytes vs 8)
  Composite:      junction tables (order_items, user_roles); natural fit
""")

## Pattern 4 — Wide Tables vs EAV vs JSON

In [ ]:
# Three ways to handle sparse/flexible attributes
# Wide table: fixed columns, NULLs for absent attrs → fast queries, inflexible schema
# EAV: entity-attribute-value → flexible but terrible query performance
# JSON column: semi-structured extra attrs → good balance for known-unknown attrs

conn3 = make_db()
conn3.executescript("""
    -- Wide table: each attribute is a column (NULLs for absent values)
    CREATE TABLE products_wide (
        product_id  INTEGER PRIMARY KEY,
        name TEXT, category TEXT, price REAL,
        color TEXT, size TEXT, weight_kg REAL,  -- clothing attrs
        voltage TEXT, wattage REAL,              -- electronics attrs
        page_count INTEGER, isbn TEXT            -- book attrs
    );
    INSERT INTO products_wide VALUES
        (1,'T-Shirt','Clothing',29.99,'Blue','M',0.2, NULL,NULL, NULL,NULL),
        (2,'Lamp','Electronics',49.99,NULL,NULL,NULL,'120V',60, NULL,NULL),
        (3,'SQL Book','Books',39.99,NULL,NULL,NULL,NULL,NULL,450,'978-3-16');

    -- EAV: entity-attribute-value (flexible but painful to query)
    CREATE TABLE product_attrs (
        product_id INTEGER, attr_name TEXT, attr_value TEXT,
        PRIMARY KEY (product_id, attr_name)
    );
    INSERT INTO product_attrs VALUES
        (1,'color','Blue'),(1,'size','M'),(1,'weight_kg','0.2'),
        (2,'voltage','120V'),(2,'wattage','60'),
        (3,'page_count','450'),(3,'isbn','978-3-16');

    -- JSON column (SQLite supports json_extract)
    CREATE TABLE products_json (
        product_id INTEGER PRIMARY KEY,
        name TEXT, category TEXT, price REAL,
        attrs TEXT  -- JSON blob for flexible attrs
    );
    INSERT INTO products_json VALUES
        (1,'T-Shirt','Clothing',29.99,'{"color":"Blue","size":"M","weight_kg":0.2}'),
        (2,'Lamp','Electronics',49.99,'{"voltage":"120V","wattage":60}'),
        (3,'SQL Book','Books',39.99,'{"page_count":450,"isbn":"978-3-16"}');
""")

run(conn3, "SELECT * FROM products_wide", "Wide table (NULLs for absent attributes)")

print("\nEAV: to get one product's attrs requires pivot:")
run(conn3, """
    SELECT product_id,
        MAX(CASE WHEN attr_name='color' THEN attr_value END) AS color,
        MAX(CASE WHEN attr_name='size'  THEN attr_value END) AS size,
        MAX(CASE WHEN attr_name='voltage' THEN attr_value END) AS voltage
    FROM product_attrs
    GROUP BY product_id
""", "EAV pivot (one GROUP BY + CASE per attribute — painful)")

run(conn3, """
    SELECT name, category, price,
        json_extract(attrs, '$.color')      AS color,
        json_extract(attrs, '$.voltage')    AS voltage,
        json_extract(attrs, '$.page_count') AS pages
    FROM products_json
""", "JSON column: flexible attrs + typed extraction")

print("""
Comparison:
  Wide:  simple queries, type safety, hard to add columns without migration
  EAV:   maximally flexible, but pivot query for every attribute, no type safety
  JSON:  flexible without pivot, partial type safety, less queryable for analytics
  Rule: wide for known-stable attrs; JSON for optional extras; avoid EAV in production
""")

## Pattern 5 — Audit / Temporal Tables

In [ ]:
# Three audit patterns: soft delete, append-only, SCD Type 2

conn4 = make_db()
conn4.executescript("""
    -- Soft delete: is_deleted flag, never physically remove rows
    CREATE TABLE customers_soft (
        customer_id INTEGER PRIMARY KEY,
        name TEXT, email TEXT,
        is_deleted INTEGER DEFAULT 0,
        deleted_at TEXT
    );
    INSERT INTO customers_soft VALUES
        (1,'Alice','a@e.com',0,NULL),
        (2,'Bob','b@e.com',1,'2024-02-15'),  -- soft-deleted
        (3,'Carol','c@e.com',0,NULL);

    -- SCD Type 2: full history with valid_from / valid_to / is_current
    CREATE TABLE customers_scd (
        scd_id      INTEGER PRIMARY KEY,
        customer_id INTEGER,
        name TEXT, email TEXT, tier TEXT,
        valid_from  TEXT,
        valid_to    TEXT,   -- NULL = current record
        is_current  INTEGER
    );
    INSERT INTO customers_scd VALUES
        (1,101,'Alice','a@e.com','Silver','2022-01-01','2023-06-15',0),
        (2,101,'Alice','a@e.com','Gold',  '2023-06-15',NULL,        1),
        (3,102,'Bob',  'b@e.com','Bronze','2021-09-01','2024-01-10',0),
        (4,102,'Bob',  'b@e.com','Silver','2024-01-10',NULL,        1);

    -- Audit log: separate table for change tracking
    CREATE TABLE customers_audit (
        audit_id     INTEGER PRIMARY KEY,
        customer_id  INTEGER,
        action       TEXT,  -- INSERT / UPDATE / DELETE
        changed_by   TEXT,
        changed_at   TEXT,
        old_tier     TEXT,
        new_tier     TEXT
    );
    INSERT INTO customers_audit VALUES
        (1,101,'UPDATE','admin','2023-06-15','Silver','Gold'),
        (2,102,'UPDATE','system','2024-01-10','Bronze','Silver');
""")

# Soft delete: only show active records
run(conn4, "SELECT * FROM customers_soft WHERE is_deleted = 0", "Soft delete: active customers only")

# SCD Type 2: current state
run(conn4, "SELECT customer_id, name, tier, valid_from FROM customers_scd WHERE is_current = 1",
    "SCD Type 2: current tier per customer")

# SCD Type 2: point-in-time query (what was Alice's tier on 2022-12-01?)
run(conn4, """
    SELECT customer_id, name, tier, valid_from, valid_to
    FROM customers_scd
    WHERE customer_id = 101
      AND valid_from <= '2022-12-01'
      AND (valid_to IS NULL OR valid_to > '2022-12-01')
""", "SCD Type 2: point-in-time query (Alice's tier on 2022-12-01)")

run(conn4, "SELECT * FROM customers_audit ORDER BY changed_at", "Audit log")

## Full Decision Map

```
SCHEMA DESIGN DECISION TREE
────────────────────────────
Workload type?
  OLTP (high write, row access)   → normalize to 3NF+, surrogate keys, tight indexes
  Analytics (high read, scans)    → denormalize, wide tables, star/snowflake schema

Primary key selection:
  Stable identifier exists?       → natural key (ISBN, SSN, IATA code)
  Identifier can change?          → surrogate int + unique constraint on natural
  Distributed / cross-service?    → UUID
  Junction / linking table?       → composite key of FKs

Flexible attributes:
  Known, stable attrs?            → wide table columns
  Optional extras, few queryable? → JSON column
  Many dynamic attrs?             → separate attr table (avoid EAV anti-pattern)

Change tracking:
  Don't delete, just mark gone?   → soft delete (is_deleted + deleted_at)
  Need full history of changes?   → SCD Type 2 (valid_from/to + is_current)
  Who changed what when?          → audit_log table (action, changed_by, before/after)

Normalization guidelines:
  1NF: one value per cell, no repeating groups
  2NF: no partial key dependency (matters with composite PKs)
  3NF: no transitive dependency (non-key depends only on key)
  Denorm: OK for reporting layers, summary tables, pre-computed aggregates
```

## Cheat Sheet

```sql
-- Normal form violations:
-- 1NF: multi-value → split to separate rows
-- 2NF: product_name in order_items (depends on product_id, not full key) → move to products
-- 3NF: zip → city (city depends on zip, not customer_id) → zip_codes table

-- Soft delete
ALTER TABLE customers ADD COLUMN is_deleted INTEGER DEFAULT 0;
ALTER TABLE customers ADD COLUMN deleted_at TEXT;
UPDATE customers SET is_deleted=1, deleted_at=CURRENT_TIMESTAMP WHERE customer_id=?;
SELECT * FROM customers WHERE is_deleted = 0;  -- active only

-- SCD Type 2: expire old, insert new
UPDATE customers_scd SET valid_to=CURRENT_DATE, is_current=0
  WHERE customer_id=? AND is_current=1;
INSERT INTO customers_scd(customer_id, tier, valid_from, valid_to, is_current)
  VALUES (?, 'Gold', CURRENT_DATE, NULL, 1);

-- Point-in-time query (SCD Type 2)
SELECT * FROM customers_scd
WHERE customer_id=101
  AND valid_from <= '2023-01-01'
  AND (valid_to IS NULL OR valid_to > '2023-01-01');

-- JSON attr extraction (SQLite / PostgreSQL)
SELECT name, json_extract(attrs, '$.color') AS color FROM products;  -- SQLite
SELECT name, attrs->>'color' AS color FROM products;                  -- PostgreSQL

-- Composite PK for junction table
CREATE TABLE order_items(
  order_id INTEGER REFERENCES orders(id),
  product_id INTEGER REFERENCES products(id),
  qty INTEGER NOT NULL,
  PRIMARY KEY (order_id, product_id)
);
```

## Summary Map

```
SQL SCHEMA DESIGN — ONE-PAGE SUMMARY
──────────────────────────────────────

NORMAL FORMS
  1NF: atomic values + no repeating groups + primary key
  2NF: 1NF + non-key cols depend on entire key (not partial)
  3NF: 2NF + non-key cols depend only on key (no transitive)
  BCNF: every determinant is a candidate key

PRIMARY KEYS
  Surrogate int: stable, small, join-friendly — default choice for OLTP
  Natural key:   use when business enforces uniqueness and it won't change
  UUID:          distributed systems, globally unique, 16 bytes
  Composite:     junction tables — (order_id, product_id) as PK

FLEXIBLE ATTRIBUTES
  Wide table: best performance, schema changes need migration
  JSON column: flexible extras, limited query support
  EAV: avoid — pivot queries for every attribute, no type safety

CHANGE TRACKING
  Soft delete:   is_deleted + deleted_at; simple; WHERE is_deleted=0 everywhere
  SCD Type 2:    valid_from/to + is_current; point-in-time queries
  Audit log:     separate table; action + before/after + who/when

INTERVIEW SIGNALS
  ✓ Explain 1NF/2NF/3NF with concrete violation examples
  ✓ Surrogate vs natural key trade-offs
  ✓ SCD Type 2 point-in-time query pattern
  ✓ Soft delete and impact on queries (always add WHERE is_deleted=0)
  ✓ When to denormalize (analytics, rarely-changing attrs)
```